### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "SMOTE" / "base_predictors" / "RandomForest_thres0.5_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.95,,,,,
1,0,cf_1,,,,,,,17.4543,,,0.0663,1,False,0.1667,0.1533
2,0,cf_2,,,,7,,,18.9745,,,0.1255,2,False,0.1667,0.27
3,0,cf_3,3,,,,,,18.3672,,,0.0685,2,False,0.1667,0.18
4,0,cf_4,,,,6,,,18.3014,,,0.113,2,False,0.1667,0.3067
5,0,cf_5,1,,,,,,16.0983,,,0.25,2,False,0.1667,0.2733
6,0,cf_6,,,6,7,,,18.9745,,,0.2505,3,False,0.1667,0.1833
7,0,cf_7,,,,7,,,18.9745,7,,0.2505,3,True,0.1667,0.0567
8,0,cf_8,,,,6,1,,18.9619,,,0.2094,3,False,0.1667,0.2567
9,0,cf_9,2,,,,,,18.9619,3,,0.138,3,True,0.1667,0.06


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.95,,,,,
9,0,cf_7,,,,7,,,18.9745,7,,0.2505,3,True,0.1667,0.0567
10,0,cf_9,2,,,,,,18.9619,3,,0.138,3,True,0.1667,0.06
11,0,cf_10,,,5,,,,18.9619,2,,0.1201,3,True,0.1667,0.0367
1,1,xin,3,4,1,2,3,0,22.3757,0,62.08,,,,,
12,1,cf_1,,,4,,,,22.3757,4,,0.2011,3,True,0.52,0.1967
13,1,cf_2,,,,3,1,,22.3757,,,0.1546,3,True,0.52,0.02
14,1,cf_4,,2,,6,,,22.3757,,,0.2296,3,True,0.52,0.2333
15,1,cf_5,,2,,,,,22.3757,2,,0.1653,3,True,0.52,0.1433
16,1,cf_6,,2,,,2,,22.3757,,,0.1921,3,True,0.52,0.23


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.95,,,,,
9,0,cf_7,,,,7,,,18.9745,7,,0.2505,3,True,0.1667,0.0567
1,1,xin,3,4,1,2,3,0,22.3757,0,62.08,,,,,
10,1,cf_2,,,,3,1,,22.3757,,,0.1546,3,True,0.52,0.02
2,2,xin,5,3,1,1,4,0,22.694,7,3.02,,,,,
11,2,cf_2,1,,,4,,,22.68,,,0.1877,3,True,0.3333,0.14
3,3,xin,3,3,6,6,2,0,24.3418,1,3.31,,,,,
12,3,cf_10,,,,7,1,,24.3375,,,0.2502,3,True,0.0367,0.0033
4,4,xin,5,4,2,7,1,0,21.2585,3,3.31,,,,,
13,4,cf_2,,,6,,,,21.2585,,,0.2368,2,True,0.0067,0.0


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_7 ---
Predicted risk: 0.0567
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7

--- cf_9 ---
Predicted risk: 0.06
Valid: True
Changes:
  etfruit: 4 → 2
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 3

--- cf_10 ---
Predicted risk: 0.0367
Valid: True
Changes:
  cgtsmok: 3 → 5
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 2


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.1967
Valid: True
Changes:
  cgtsmok: 1 → 4
  dosprt: 0 → 4

--- cf_2 ---
Predicted risk: 0.02
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_4 ---
Predicted risk: 0